# NBA season futures: pricing pipeline

One as-of date in, four contract types out. Run top to bottom.

Calibration of the carryover prior lives in `calibration.ipynb`; the numbers it
produced are stored in `leagues/nba.py` and only read here.


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Walk up until the project root is found. Keying on the engine package rather
# than on a data folder matters: a notebook folder can acquire its own data
# directory, which would stop the walk one level too early.
ROOT = Path.cwd()
while not (ROOT / "engine").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from leagues.nba import NBA
from engine.observations import split_at_asof, build_observations
from engine.ratings import fit_ratings
from engine.carryover import Carryover, build_prior, prior_weight
from engine.probabilities import win_probability, schedule_win_probabilities
from engine.simulate import simulate_remaining_season, season_win_total_fair
from engine.contracts import simulate_full_seasons, contract_prices

DATA = ROOT / "data" / "processed"
games = pd.read_csv(DATA / "games.csv", parse_dates=["date"])

print("ROOT:   ", ROOT)
print("games:  ", len(games), "rows,", games["season"].nunique(), "seasons")
print("columns:", games.columns.tolist())

ROOT:    c:\Projects\FuturesTakeHomeAssignmentPP
games:   27065 rows, 21 seasons
columns: ['season', 'game_id', 'date', 'season_type', 'neutral_site', 'away_team', 'home_team', 'away_points', 'home_points', 'home_margin', 'home_win', 'overtime', 'is_playoff', 'is_playin', 'atypical_season']


## 2. Inputs

The only two things a trader changes: which season is being priced and what
today is.

In [2]:
SEASON = 2024
ASOF = "2024-01-15"

## 3. Prior

Team strength before a ball is bounced. Last season's final ratings are pulled
toward the league mean by the carryover factor, and the prior weight says how
much evidence that belief is worth: the square of the weight is the number of
games the prior stands in for. Both numbers are calibrated in
`calibration.ipynb` and stored in the league config.

In [3]:
carryover = Carryover(NBA.carryover_beta, NBA.carryover_tau, n_pairs=0)

# The team list comes from the schedule, not from played games: at the start of
# a season nothing has been played and the model still has to produce prices.
regular = games[
    (games["season"] == SEASON) & (games["season_type"] == "Regular Season")
]
teams = sorted(set(regular["home_team"]) | set(regular["away_team"]))

previous = games[
    (games["season"] == SEASON - 1) & (games["season_type"] == "Regular Season")
]
previous_played, _ = split_at_asof(
    games, season=SEASON - 1, asof=previous["date"].max() + pd.Timedelta(days=1)
)
previous_ratings = fit_ratings(build_observations(previous_played)).ratings

prior = build_prior(previous_ratings, carryover, teams)
weight = prior_weight(NBA.margin_sd, carryover)

print(f"prior weight {weight:.2f}, worth {weight ** 2:.1f} games")
print(prior.sort_values(ascending=False).head(5))

prior weight 3.61, worth 13.0 games
BOS    3.590794
CLE    2.789910
PHI    2.460578
MIL    2.356789
MEM    2.105487
dtype: float64


## 4. Split the season at the as-of date

Everything before today is a fact and is never resimulated. Everything from
today onward is what gets played out.

In [4]:
played, remaining = split_at_asof(games, season=SEASON, asof=ASOF)
obs = build_observations(played)

print(f"played {len(played)}, remaining {len(remaining)}, total {len(played) + len(remaining)}")
print(obs["observed_margin"].describe())

played 584, remaining 646, total 1230
count    584.000000
mean       2.250000
std       13.417271
min      -25.000000
25%       -8.000000
50%        3.000000
75%       12.000000
max       25.000000
Name: observed_margin, dtype: float64


## 5. Team strength

Ratings are in points: a rating of 9 means a team beats a league-average
opponent by nine on a neutral floor. The prior enters as extra rows in the same
least-squares problem, so early in a season it dominates and by January it does
not.

In [5]:
result = fit_ratings(obs, prior_ratings=prior, prior_weight=weight, teams=teams)

print(f"home advantage: {result.home_advantage:.2f} points")
print(f"residual sd:    {result.residual_sd:.2f} points")
print(result.ratings.head(8))
print(result.ratings.tail(5))

home advantage: 2.01 points
residual sd:    11.83 points
BOS    7.695905
OKC    5.487149
MIN    4.992043
PHI    4.985421
LAC    3.993071
DEN    3.791737
NYK    3.572891
NOP    3.410094
dtype: float64
POR   -6.194197
WAS   -6.273215
SAS   -6.698674
CHA   -8.190792
DET   -8.718581
dtype: float64


## 6. Game probabilities

The bridge from points to wins. Residual spread is what stops a nine-point
favourite from winning every time.

In [6]:
for m in [0, result.home_advantage, 5, 10, 19.5]:
    print(f"expected margin {m:5.2f}  ->  {win_probability(m, result.residual_sd):.3f}")

probs = schedule_win_probabilities(
    remaining, result.ratings, result.home_advantage, result.residual_sd
)
print(probs.describe())

# Calibration check: the model's average home win rate on games already played
# should match what actually happened.
fitted_probs = schedule_win_probabilities(
    played, result.ratings, result.home_advantage, result.residual_sd
)
print(f"\nobserved home win share: {played['home_win'].mean():.3f}")
print(f"fitted mean on played:   {fitted_probs.mean():.3f}")

expected margin  0.00  ->  0.500
expected margin  2.01  ->  0.568
expected margin  5.00  ->  0.664
expected margin 10.00  ->  0.801
expected margin 19.50  ->  0.950
count    646.000000
mean       0.554570
std        0.178970
min        0.111761
25%        0.421464
50%        0.565051
75%        0.691471
max        0.940313
dtype: float64

observed home win share: 0.579
fitted mean on played:   0.568


## 7. Regular season: win totals

Contract type four. Total wins across the league is the cheapest sanity check
there is: it has to come out at the number of games in the season.

In [7]:
sims = simulate_remaining_season(played, remaining, probs, n_sims=20_000)

projected = sims.mean().sort_values(ascending=False)
print(projected.head(6))
print(projected.tail(4))
print(f"\ntotal wins across league: {sims.sum(axis=1).mean():.1f}")

print(season_win_total_fair(sims, "BOS", 58.5))
print(season_win_total_fair(sims, "DET", 20.5))

BOS    62.36960
MIN    57.40770
OKC    56.63915
PHI    53.52500
DEN    53.22915
MIL    52.72870
dtype: float64
WAS    20.90885
CHA    19.41155
SAS    19.36860
DET    13.26250
dtype: float64

total wins across league: 1230.0
{'team': 'BOS', 'line': 58.5, 'over': 0.9171, 'under': 0.0829, 'push': 0.0, 'mean_wins': 62.3696, 'median_wins': 62.0, 'sd_wins': 2.7423114071149324}
{'team': 'DET', 'line': 20.5, 'over': 0.0055, 'under': 0.9945, 'push': 0.0, 'mean_wins': 13.2625, 'median_wins': 13.0, 'sd_wins': 2.6963600033866757}


## 8. Full season: all four contracts

Regular season and postseason in one loop, because the bracket depends on the
standings. The three sums at the bottom are structural identities: if any of
them is off, something upstream is broken.

In [8]:
from engine.uncertainty import strength_paths

N_SIMS = 5_000
rng = np.random.default_rng(1)
STEP_DAYS = 7

# Map every remaining game and every playoff round onto a step of the walk.
# The regular season ends when the schedule ends; the postseason after it runs
# roughly a week to the play-in and two and a half weeks per round.
asof = pd.Timestamp(ASOF)
game_step = ((remaining["date"] - asof).dt.days // STEP_DAYS).to_numpy()
end_step = int(((remaining["date"].max() - asof).days // STEP_DAYS))
round_step = [end_step + 1 + int(2.5 * r) for r in range(len(NBA.round_names))]
n_steps = round_step[-1] + 2

paths, path_teams = strength_paths(
    result, N_SIMS, n_steps, rng, NBA.daily_strength_vol, STEP_DAYS
)

spread = paths.std(axis=0)
print(f"steps: {n_steps}, season ends at {end_step}, rounds at {round_step}")
print(f"strength sd today:        {spread[0].mean():.2f} points")
print(f"at the end of the season: {spread[end_step].mean():.2f} points")
print(f"at the finals:            {spread[round_step[-1]].mean():.2f} points")

results = simulate_full_seasons(
    played, remaining, probs,
    result.ratings, result.home_advantage, result.residual_sd,
    NBA, n_sims=N_SIMS,
    strength_paths=paths, path_teams=path_teams,
    game_step=game_step, round_step=round_step,
)

prices = contract_prices(results)
print(prices.head(10).to_string(index=False))

print(f"\nchampionship probabilities sum to {prices['win_championship'].sum():.4f}")
print(f"playoff berths sum to {prices['make_playoffs'].sum():.2f}")
print(f"conference finals berths sum to {prices['reach_conference_finals'].sum():.2f}")

steps: 22, season ends at 12, rounds at [13, 15, 18, 20]
strength sd today:        1.60 points
at the end of the season: 2.77 points
at the finals:            3.32 points
team  make_playoffs  win_championship  mean_wins  sd_wins  reach_conference_semifinals  reach_conference_finals  reach_finals
 BOS         1.0000            0.2802    62.0138 3.546114                       0.8746                   0.6588        0.4434
 OKC         0.9988            0.1314    56.4056 3.905347                       0.7370                   0.4470        0.2534
 MIN         0.9990            0.1058    57.1880 3.873094                       0.7210                   0.4226        0.2282
 PHI         0.9926            0.1032    53.3984 4.108948                       0.7082                   0.3976        0.1936
 LAC         0.9784            0.0626    51.4932 4.075939                       0.5618                   0.2818        0.1422
 DEN         0.9912            0.0608    53.0372 3.869006                

In [9]:
from leagues.nfl import NFL
from engine.playoffs import simulate_bracket

# The postseason machinery is the part a second league actually exercises: byes,
# reseeding and single-game rounds are all new paths. Run the bracket on a
# synthetic seeded field to check they work before claiming the engine is general.
rng = np.random.default_rng(0)
afc = [t for t, c in NFL.conference.items() if c == "AFC"][:7]
nfc = [t for t, c in NFL.conference.items() if c == "NFC"][:7]
fake_ratings = {t: 6.0 - i for i, t in enumerate(afc + nfc)}
fake_wins = {t: 14 - i for i, t in enumerate(afc + nfc)}

champions = []
for _ in range(2000):
    reached = simulate_bracket(
        {"AFC": afc, "NFC": nfc},
        [fake_ratings] * (len(NFL.round_names) + 1),
        fake_wins, 2.0, 13.5, NFL, rng, use_qualification=True,
    )
    champions.append(max(reached, key=reached.get))

counts = pd.Series(champions).value_counts(normalize=True)
print(counts.round(3).to_string())
print(f"\nteams reaching the postseason: {len(reached)}")

BAL    0.335
BUF    0.150
ARI    0.110
CIN    0.108
CLE    0.071
ATL    0.055
DEN    0.042
HOU    0.032
CAR    0.030
CHI    0.022
IND    0.018
DAL    0.014
DET    0.006
GB     0.004

teams reaching the postseason: 14


## Demo season

In [10]:
# The same model at three points in a season. Nothing changes between runs except
# the as-of date: at the opening the prior carries everything, by January the
# season has taken over, and by late March the ratings are as settled as they get
# while the postseason is still five weeks of uncertainty away.
DEMO_SEASON = 2026
demo_regular = games[
    (games["season"] == DEMO_SEASON) & (games["season_type"] == "Regular Season")
]
demo_teams = sorted(set(demo_regular["home_team"]) | set(demo_regular["away_team"]))

demo_previous, _ = split_at_asof(
    games, season=DEMO_SEASON - 1,
    asof=games[(games["season"] == DEMO_SEASON - 1)
               & (games["season_type"] == "Regular Season")]["date"].max()
         + pd.Timedelta(days=1),
)
demo_prior = build_prior(
    fit_ratings(build_observations(demo_previous)).ratings, carryover, demo_teams
)

rows = []
for label, asof_date in [
    ("opening night", demo_regular["date"].min()),
    ("mid January", pd.Timestamp(f"{DEMO_SEASON}-01-15")),
    ("late March", pd.Timestamp(f"{DEMO_SEASON}-03-25")),
]:
    demo_played, demo_remaining = split_at_asof(games, season=DEMO_SEASON, asof=asof_date)
    demo_fit = fit_ratings(
        build_observations(demo_played), prior_ratings=demo_prior,
        prior_weight=weight, teams=demo_teams,
    )

    demo_rng = np.random.default_rng(1)
    demo_probs = schedule_win_probabilities(
        demo_remaining, demo_fit.ratings, demo_fit.home_advantage, demo_fit.residual_sd
    )
    demo_step = ((demo_remaining["date"] - asof_date).dt.days // STEP_DAYS).to_numpy()
    demo_end = int(demo_step.max())
    demo_rounds = [demo_end + 1 + int(2.5 * r) for r in range(len(NBA.round_names))]

    demo_paths, demo_path_teams = strength_paths(
        demo_fit, N_SIMS, demo_rounds[-1] + 2, demo_rng,
        NBA.daily_strength_vol, STEP_DAYS,
    )
    demo_results = simulate_full_seasons(
        demo_played, demo_remaining, demo_probs,
        demo_fit.ratings, demo_fit.home_advantage, demo_fit.residual_sd,
        NBA, n_sims=N_SIMS,
        strength_paths=demo_paths, path_teams=demo_path_teams,
        game_step=demo_step, round_step=demo_rounds, teams=demo_teams,
    )
    demo_prices = contract_prices(demo_results).set_index("team")

    top = demo_prices["win_championship"].idxmax()
    rows.append({
        "as of": label,
        "games played": len(demo_played),
        "favourite": top,
        "title": round(float(demo_prices.loc[top, "win_championship"]), 3),
        "mean wins": round(float(demo_prices.loc[top, "mean_wins"]), 1),
        "sd wins": round(float(demo_prices.loc[top, "sd_wins"]), 2),
        "rating sd": round(float(np.sqrt(np.diag(demo_fit.rating_cov)).mean()), 2),
    })

print(pd.DataFrame(rows).to_string(index=False))

        as of  games played favourite  title  mean wins  sd wins  rating sd
opening night             0       OKC  0.213       59.2     8.96       3.00
  mid January           602       OKC  0.402       64.5     3.47       1.65
   late March          1079       OKC  0.416       64.2     1.42       1.31


In [11]:
# The starting point the rest of the model is measured against: point estimates,
# no prior, strength held fixed for the whole season. Everything after this is a
# correction to it.
naive_fit = fit_ratings(obs)
naive_probs = schedule_win_probabilities(
    remaining, naive_fit.ratings, naive_fit.home_advantage, naive_fit.residual_sd
)
naive_results = simulate_full_seasons(
    played, remaining, naive_probs,
    naive_fit.ratings, naive_fit.home_advantage, naive_fit.residual_sd,
    NBA, n_sims=N_SIMS, teams=teams,
)
naive_prices = contract_prices(naive_results).set_index("team")

top = naive_prices["win_championship"].idxmax()
print(f"no prior, no uncertainty: {top} title {naive_prices.loc[top, 'win_championship']:.3f}, "
      f"sd wins {naive_prices.loc[top, 'sd_wins']:.2f}")

no prior, no uncertainty: BOS title 0.489, sd wins 2.59


In [12]:
# The title price is more sensitive to this parameter than to anything else, so
# the range between the two defensible calibrations is worth quoting directly.
for vol in [0.185, NBA.daily_strength_vol]:
    vol_rng = np.random.default_rng(1)
    vol_paths, vol_teams = strength_paths(
        result, N_SIMS, n_steps, vol_rng, vol, STEP_DAYS
    )
    vol_results = simulate_full_seasons(
        played, remaining, probs,
        result.ratings, result.home_advantage, result.residual_sd,
        NBA, n_sims=N_SIMS, teams=teams,
        strength_paths=vol_paths, path_teams=vol_teams,
        game_step=game_step, round_step=round_step,
    )
    vol_prices = contract_prices(vol_results).set_index("team")
    best = vol_prices["win_championship"].idxmax()
    print(f"vol {vol:.3f}: {best} title {vol_prices.loc[best, 'win_championship']:.3f}, "
          f"sd wins {vol_prices.loc[best, 'sd_wins']:.2f}")

vol 0.185: BOS title 0.316, sd wins 3.42
vol 0.250: BOS title 0.280, sd wins 3.55


# Schedule 2026-2027

In [13]:
from nba_api.stats.endpoints import scheduleleaguev2

UPCOMING = 2027   # season ending in 2027, i.e. 2026-27

schedule_raw = scheduleleaguev2.ScheduleLeagueV2(
    season=f"{UPCOMING - 1}-{str(UPCOMING)[-2:]}", timeout=60
).get_data_frames()[0]

# Game ids carry the season type in their first three digits: 001 preseason,
# 002 regular season, 003 all-star, 004 playoffs, 005 play-in. Filtering on the
# id is more reliable than on the labels, which are free text.
upcoming = schedule_raw[schedule_raw["gameId"].astype(str).str[:3] == "002"].copy()

upcoming = pd.DataFrame({
    "season": UPCOMING,
    "game_id": upcoming["gameId"],
    "date": pd.to_datetime(upcoming["gameDateEst"]).dt.tz_localize(None).dt.normalize(),
    "season_type": "Regular Season",
    "neutral_site": upcoming["isNeutral"].astype(bool),
    "away_team": upcoming["awayTeam_teamTricode"],
    "home_team": upcoming["homeTeam_teamTricode"],
}).sort_values("date").reset_index(drop=True)

print(f"games: {len(upcoming)}")
print(f"dates: {upcoming['date'].min().date()} to {upcoming['date'].max().date()}")
print(f"teams: {upcoming['home_team'].nunique()}")
per_team = pd.concat([upcoming["home_team"], upcoming["away_team"]]).value_counts()
print(f"games per team: {per_team.min()} to {per_team.max()}")
print(f"neutral site games: {int(upcoming['neutral_site'].sum())}")

games: 1206
dates: 2026-10-20 to 2027-04-11
teams: 30
games per team: 80 to 80
neutral site games: 3


In [14]:
missing = upcoming[upcoming["home_team"].isna() | upcoming["away_team"].isna()]
print(f"rows without both teams: {len(missing)}")
print(missing[["date", "game_id", "home_team", "away_team"]].head())

upcoming = upcoming.dropna(subset=["home_team", "away_team"]).reset_index(drop=True)
per_team = pd.concat([upcoming["home_team"], upcoming["away_team"]]).value_counts()
print(f"\ngames: {len(upcoming)}, per team {per_team.min()} to {per_team.max()}")

rows without both teams: 6
          date     game_id home_team away_team
325 2026-12-04  0022601201       NaN       NaN
326 2026-12-04  0022601202       NaN       NaN
327 2026-12-05  0022601203       NaN       NaN
328 2026-12-05  0022601204       NaN       NaN
329 2026-12-08  0022601229       NaN       NaN

games: 1200, per team 80 to 80


In [15]:
# Pricing a season that has not started. The only input is the prior: last
# season's ratings pulled toward the league mean. Nothing else in the model
# changes, which is the point of having a prior at all.
UPCOMING_PREV = UPCOMING - 1
up_teams = sorted(set(upcoming["home_team"]) | set(upcoming["away_team"]))

prev_reg = games[
    (games["season"] == UPCOMING_PREV) & (games["season_type"] == "Regular Season")
]
prev_played, _ = split_at_asof(
    games, season=UPCOMING_PREV, asof=prev_reg["date"].max() + pd.Timedelta(days=1)
)
up_prior = build_prior(
    fit_ratings(build_observations(prev_played)).ratings, carryover, up_teams
)

# With nothing played, the fit returns the prior itself, and both the team list
# and the home advantage have to come from outside the season.
up_played = upcoming.iloc[0:0].assign(
    home_margin=pd.Series(dtype=float), home_win=pd.Series(dtype=bool)
)
up_fit = fit_ratings(
    build_observations(up_played), prior_ratings=up_prior,
    prior_weight=weight, teams=up_teams, fallback_sd=NBA.margin_sd,
)
up_ha = NBA.preseason_home_advantage

up_asof = upcoming["date"].min()
up_probs = schedule_win_probabilities(
    upcoming, up_fit.ratings, up_ha, up_fit.residual_sd
)
up_step = ((upcoming["date"] - up_asof).dt.days // STEP_DAYS).to_numpy()
up_rounds = [int(up_step.max()) + 1 + int(2.5 * r) for r in range(len(NBA.round_names))]

up_rng = np.random.default_rng(1)
up_paths, up_path_teams = strength_paths(
    up_fit, N_SIMS, up_rounds[-1] + 2, up_rng, NBA.daily_strength_vol, STEP_DAYS
)
up_results = simulate_full_seasons(
    up_played, upcoming, up_probs,
    up_fit.ratings, up_ha, up_fit.residual_sd,
    NBA, n_sims=N_SIMS, teams=up_teams,
    strength_paths=up_paths, path_teams=up_path_teams,
    game_step=up_step, round_step=up_rounds,
)

up_prices = contract_prices(up_results)
print(up_prices.head(12).to_string(index=False))
print(f"\nchampionship sums to {up_prices['win_championship'].sum():.4f}")
print(f"playoff berths sum to {up_prices['make_playoffs'].sum():.2f}")
print("schedule carries 80 of 82 games per team, so win totals run about two light")

team  make_playoffs  win_championship  mean_wins  sd_wins  reach_conference_semifinals  reach_conference_finals  reach_finals
 OKC         0.9214            0.1338    53.2060 8.958237                       0.6370                   0.4014        0.2374
 SAS         0.8786            0.1014    51.1800 9.393596                       0.5766                   0.3344        0.1848
 DET         0.8340            0.0872    49.1804 9.466963                       0.5176                   0.2918        0.1574
 BOS         0.8486            0.0796    49.3804 9.581339                       0.5258                   0.2934        0.1602
 HOU         0.7814            0.0584    46.9762 9.608397                       0.4258                   0.2122        0.1082
 NYK         0.7678            0.0582    46.4574 9.656875                       0.4230                   0.2218        0.1160
 DEN         0.7636            0.0562    46.5476 9.682235                       0.4226                   0.2164       

In [16]:
prev_ratings_check = fit_ratings(build_observations(prev_played)).ratings
print(prev_ratings_check.head(10).round(2).to_string())
print(up_prior.sort_values(ascending=False).head(10).round(2).to_string())

OKC    9.32
SAS    7.78
DET    6.64
BOS    6.50
HOU    4.80
NYK    4.69
DEN    4.55
CLE    3.62
CHA    3.45
MIN    2.52
OKC    5.77
SAS    4.82
DET    4.11
BOS    4.02
HOU    2.97
NYK    2.90
DEN    2.82
CLE    2.24
CHA    2.14
MIN    1.56
